# Predicting Single-Bid Public Procurement Contracts in Bulgaria

### A competition red-flag indicator: a smoke detector, not a judge

This is the project write-up: what is being predicted, why it matters, where the
data comes from, what the model is **not** allowed to claim, and (by the end of
the project) what was found. The step-by-step analysis lives in the numbered
notebooks in `notebooks/`; this document is the front page and the conclusion.

## 1. The problem

Public procurement is how the state buys works, goods and services, and it is a
huge share of the economy. The European Commission puts public procurement at
about 15% of EU GDP, and the OECD at roughly 13% of GDP across its member
countries. So even small improvements in how competitive that spending is carry
real value for money.

A contract that attracts **only one bid** is the simplest warning sign that
competition was weak. It does not prove anything went wrong - there are perfectly
innocent reasons for a single bid (a niche supplier, a tiny local job, an urgent
need). But across many contracts, a high single-bid rate is a recognised red
flag, and it is exactly the kind of pattern that procurement watchdogs,
journalists and oversight bodies want to look at more closely.

**The question this project asks:** *given the characteristics of a contract that
are known around the time it is announced, can we predict whether it will end up
with a single bid - and which characteristics drive that pattern?*

The unit of analysis is the **contract**, not the company and not the authority.
Every row is one awarded contract (or lot).

## 2. Why single-bid is a recognised indicator

This is not a metric invented for the project. Two well-known bodies treat
single-bidding as a standard competition signal, and their framing also shows how
to be careful with it.

**European Commission - Single Market Scoreboard.** The Commission has long
tracked a "Single bidder" indicator: the proportion of awarded contracts that
received only one bid. It rates a country green at 10% or below and red above
20%, and describes the indicator as reflecting several aspects of procurement,
including competition and bureaucracy. Two methodological details matter here:

- **Framework agreements are excluded**, because they have different reporting
  patterns.
- **Direct awards are excluded** - procurement negotiated without a call for
  competition, or awarded without prior publication of a notice - because the
  rules for those procedures *make no provision for competition in the first
  place*.

That second point is important: the Commission does **not** mix direct awards
into the single-bidder measure, precisely because counting them would be circular
- a procedure with no call for bids is almost guaranteed to show one bid. This
project hits the same trap and handles it the same way.

**How does Bulgaria look on this measure?** On the EC's own Scoreboard (2024
reporting period, based on EU-wide TED data), Bulgaria's single-bidder rate is
**36%** - well above the 20% "red" threshold - and its direct-award rate is
**20%, the highest in the EU** and double the 10% red line for that indicator. So
weak competition is not a marginal concern in Bulgarian procurement; on the EU's
own scale it is among the most pronounced in the Union.

![EC Single Market Scoreboard, Indicator 1 — single-bidder rate by country, 2024](images/ec_single_bidder.png)

*EC Single Market Scoreboard, Indicator [1] Single bidder (2024 reporting period,
TED data). Bulgaria (BG) sits near the top at 36%, well into the "red" band
(threshold 20%). Source: European Commission - see References.*

![EC Single Market Scoreboard, Indicator 2 — direct-award rate by country, 2024](images/ec_direct_award.png)

*EC Single Market Scoreboard, Indicator [2] Direct awards (2024 reporting period,
TED data). Bulgaria (BG) is the highest in the EU at 20%, double the "red"
threshold of 10%. Source: European Commission - see References.*

The exploration in `01` reaches the same conclusion from the data used here. On
the SIGMA snapshot, the single-bid rate is **44.9%** across all contracts with a
known bid count, or **40.0%** computed the EC way (direct awards excluded). These
are not identical to the EC's 36% - the snapshot pools 2020–2026 rather than a
single year, draws on SIGMA rather than TED, and cannot separately exclude
framework agreements - but they point unmistakably the same way: roughly twice
the red threshold. (A robustness check in `01` confirms the figure is not an
artefact of the ambiguous "Unknown" procedure category: removing it as well moves
the rate only from 40.0% to 38.8%.)

Crucially, this is **not** just a direct-award story. Of the ~78,000 single-bid
contracts in the snapshot, about **78% are not direct awards** at all - so even
setting aside the one near-circular procedure type, single-bid remains the
typical outcome across competitive procedures. That is what makes the prediction
task worthwhile rather than tautological.

**OECD - Fighting Bid Rigging in Public Procurement.** The OECD frames bid
rigging as firms conspiring to raise prices or lower quality in public tenders,
and recommends that tenders be designed to reduce collusion risk and that
officials be trained to detect and report it. The key idea taken from this side
of the literature: red flags are **signals to investigate, not proof of
wrongdoing**, and a pattern over many contracts says more than any single case.

*Full references with links are in the README and at the end of this notebook.*

## 3. The framing: a smoke detector, not a judge

The single most important thing to be clear about - for the project and for the
defence - is what this model is for.

> **It is a smoke detector, not a judge.** It points at contracts where
> competition looks weak and says "worth a look here." It does not, and cannot,
> say that anything illegal happened.

This matters for three reasons:

- **Single bid is not corruption.** It is a competition outcome. Plenty of
  single-bid contracts are completely clean. The model predicts the *outcome*,
  not intent.
- **A useful triage tool needs precision, not just coverage.** Whoever uses these
  flags - an auditor, a journalist - cannot investigate tens of thousands of
  contracts, so the flags need to be *meaningful*: precision matters more than
  catching every last case. This shapes the evaluation later (precision / recall
  / F1 / ROC-AUC, not bare accuracy).
- **It complements the existing register rather than repeating it.** The source
  register already shows, after the fact, which contracts had one bid. The point
  of a model is to learn *why* it happens from contract characteristics, so the
  same logic could in principle score a newly announced procedure *before*
  bidding closes - while the outcome is still unknown.

## 4. The main methodological risk, stated up front

One feature, `procedure`, carries a near-circular relationship with the target.
Some procedure types - above all "Пряко / без обявление" (direct award) - are
*defined* as effectively single-source. For those contracts a single bid is
almost structural, so a model using `procedure` is partly reading the answer off
the label rather than predicting a behaviour.

This is the project's **headline limitation**, and it is faced directly:

- measured in the exploration (`notebooks/01`): on rows with a known bid count,
  the direct-award procedure is far above every competitive one;
- investigated in the modelling stage by running the models **with and without**
  that procedure, so its effect on the scores is visible;
- written up in the Limitations section at the end.

The EC methodology gives cover: the Commission *also* keeps direct awards out of
its single-bidder indicator, for the same reason.

## 5. The data

**Source.** Around 193,000 awarded contracts from the Bulgarian public
procurement register (АОП / ЦАИС ЕОП), accessed through the SIGMA open-data
project (sigma.midt.bg). Licensed **CC-BY 4.0**, so reusable with attribution.

**What one row is.** One awarded contract (or lot): the contracting authority,
the contractor, the contract value in EUR, the CPV sector code (the EU's Common
Procurement Vocabulary classification of *what* is bought), the procedure type,
whether it is EU-funded, the signing date, and the number of bids received.

**The target.** `single_bid = (bids_received == 1)` - binary, one row per
contract.

**Features** (known at or near announcement time): `value_eur`, `sector_code`,
`kind` (supplies / services / works), `procedure`, `eu_funded`, date parts from
`signed_at`, plus two attributes joined from the authorities table - `type_group`
and `region`.

**Deliberately excluded, to avoid leakage:** anything known only *after* bidding
closes. The contractor's identity is known only post-award, so using it would
break the "score before bidding closes" framing. Aggregate per-authority columns
(total spend, contract / supplier counts, averages) are excluded too, because
they summarise outcomes that overlap with the target.

The cell below reads the **raw** contracts file directly and computes the
headline figures, so this notebook stands on its own (it does not depend on the
cleaning notebook having run).

In [2]:
import pandas as pd

# Run from repo root, so the path is data/raw/ (not ../data/raw/).
contracts = pd.read_csv("data/raw/sigma-contracts.csv")

n_total = len(contracts)
known = contracts[contracts["bids_received"].notna()]
n_known = len(known)
single_rate = (known["bids_received"] == 1).mean()

print(f"contracts in snapshot:        {n_total:,}")
print(f"with a recorded bid count:    {n_known:,}")
print(f"single-bid rate (known rows): {single_rate:.1%}")
print(f"=> target is well balanced (~45/55), so accuracy alone would mislead")

contracts in snapshot:        193,161
with a recorded bid count:    174,119
single-bid rate (known rows): 44.9%
=> target is well balanced (~45/55), so accuracy alone would mislead


## 6. Scope and assumptions

- **Conditional on a recorded bid count.** Contracts with no bid count (about 10%
  of rows) cannot be given a target and are dropped. That missingness is not
  perfectly random - it leans towards certain procedures - so the results
  describe "contracts where a bid count was recorded," not literally every
  contract. (Shown in `notebooks/02`.)
- **Outcome, not intent.** The label is "exactly one bid," nothing more. No claim
  about legality or collusion is attached to it.
- **Bulgaria, 2020–2026, as recorded.** Findings are specific to this register
  and period. A couple of stray out-of-range dates are flagged for the
  feature-engineering step.
- **Course-taught methods only.** pandas, scikit-learn, matplotlib: baseline →
  logistic regression → decision tree → random forest / boosting, evaluated
  honestly.

## 7. How the project proceeds

- `notebooks/01-initial-exploration.ipynb` - first exploration: target balance,
  the procedure trap, value skew.
- `notebooks/02-cleaning-and-joins.ipynb` - cleaning and the authority join,
  producing the modelling table in `data/processed/`.
- *(later notebooks)* - EDA, naive baseline, logistic regression, then decision
  tree and random forest / boosting.

The model spine moves from a naive baseline (which any real model has to beat) up
to ensemble methods, each compared on precision, recall, F1 and ROC-AUC rather
than accuracy alone.

## 8. Results and conclusions

*To be completed at the end of the project.* This section will summarise:

- how each model scored against the naive baseline (precision / recall / F1 /
  ROC-AUC);
- the effect of including vs. excluding the direct-award procedure (the
  circularity check);
- which features most strongly predict single-bid;
- what the model can and cannot be used for, and the practical next step
  (scoring newly announced procedures before bidding closes).

## References

- **European Commission - Single Market Scoreboard, "Access to public
  procurement"** (Single bidder and Direct awards indicators; framework
  agreements and direct awards excluded from the single-bidder measure; green
  ≤10%, red >20%).
  https://single-market-scoreboard.ec.europa.eu/business-framework-conditions/public-procurement_en

- **OECD - Fighting bid rigging in public procurement** (red flags as signals to
  investigate, not proof; tender design and detection).
  https://www.oecd.org/en/topics/sub-issues/competition-enforcement/fighting-bid-rigging-in-public-procurement.html

- **Data:** Bulgarian public procurement register (АОП / ЦАИС ЕОП) via the SIGMA
  open-data project, https://sigma.midt.bg - licensed CC-BY 4.0.